In [1]:
import os
import torch
import gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
max_seq_length = 512

print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print(f"Total VRAM: {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")


Free VRAM: 15.53 GB
Total VRAM: 15.64 GB


In [2]:
!pip install datasets

In [3]:
import transformers
transformers.__version__

'5.0.0'

In [4]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Import Dataset

In [5]:
from datasets import load_dataset,Dataset, DatasetDict

In [6]:
!pip install py7zr

### Load dataset

In [7]:
raw_data=load_dataset("cnn_dailymail", "3.0.0")
raw_data

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [8]:
# 10K train, 1K validation — perfect for 1 hour
data = DatasetDict({
    "train"     : raw_data["train"].select(range(2500)),
    "validation": raw_data["validation"].select(range(250)),
})
data

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 2500
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 250
    })
})

In [9]:
print(".......................ARTICLES........................:\n", data['train']['article'][1])
print("..................HIGHLIGHTS...........................:\n", data['train']['highlights'][1])

.......................ARTICLES........................:
 Editor's note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events. Here, Soledad O'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial. MIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor." Here, inmates with the most severe mental illnesses are incarcerated until they're ready to appear in court. Most often, they face drug charges or charges of assaulting an officer --charges that Judge Steven Leifman says are usually "avoidable felonies." He says the arrests often result from confrontations with police. Mentally ill people often won't do what they're told when police arrive on the scene -- confrontation seems to exacerbate their illness and they becom

## Data Ingestion
### Modify the data for summarization task

In [10]:
def create_prompt(example):
    example["text"] = (
        "###Human: Read this following Document and provide a clear point-by-point summary of key information.\n"
        + example["article"]
        + "\n\n###Assistant:\n"
        + example["highlights"]
    )
    return example

data = data.map(create_prompt)
print(data["train"][0]["text"])

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

###Human: Read this following Document and provide a clear point-by-point summary of key information.
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel

In [11]:
print(data)
data = data.remove_columns(["article", "highlights", "id"])
print(data)

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id', 'text'],
        num_rows: 2500
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id', 'text'],
        num_rows: 250
    })
})
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2500
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 250
    })
})


## Install unsloth

In [12]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [13]:
from unsloth import FastLanguageModel

/tmp/ipykernel_35232/2814113929.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer


In [14]:
import torch
max_seq_length = 512
dtype = None
load_in_4bit = True  # Use 4bit quantization to reduce memory usage.

## Loading the model and tokenizer

In [15]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.3.10: Fast Mistral patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [16]:
# Verify immediately after loading
print(f"Unsloth max_seq_length: {model.max_seq_length}")  # must print 512
print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

Unsloth max_seq_length: 512
Free VRAM: 11.27 GB


In [17]:
print(torch.cuda.memory_summary(abbreviated=True))


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   3988 MiB |   4004 MiB |   7841 MiB |   3852 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   3988 MiB |   4004 MiB |   7841 MiB |   3852 MiB |
|---------------------------------------------------------------------------|
| Requested memory      |   3988 MiB |   4004 MiB |   7841 MiB |   3852 MiB |
|---------------------------------------------------------------

In [18]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096, padding_idx=770)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_lay

In [19]:
tokenizer

TokenizersBackend(name_or_path='unsloth/mistral-7b-instruct-v0.3-bnb-4bit', vocab_size=32768, model_max_length=32768, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '[control_768]'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[INST]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[/INST]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("[TOOL_CALLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("[AVAILABLE_TOOLS]", rstrip=False, lstrip=False, single_w

## Setting up the peft model

In [20]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",

    use_gradient_checkpointing="unsloth",
    random_state=1000,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

Unsloth 2026.3.10 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


### Define Training Arguments

In [22]:
from unsloth import is_bfloat16_supported
from trl import SFTConfig
training_arguments = SFTConfig(
        #Batch settings
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,

        #Full training
        num_train_epochs=2,
        max_seq_length = 512,

        #Learning Rate
        warmup_steps = 50,
        learning_rate = 2e-5,
        lr_scheduler_type = "cosine",

        #Precision
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),

        #Memory Saving
        optim = "adamw_8bit",
        weight_decay = 0.01,

        #Logging and saving
        logging_steps = 25,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        load_best_model_at_end = True,
        save_total_limit = 1,

        #Misc
        seed = 1000,
        output_dir = "outputs",
        logging_dir ="./logs",
        report_to = "none"
    )

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### Define trainer

In [23]:
from trl import SFTTrainer
trainer = SFTTrainer(
model = model,
    tokenizer = tokenizer,
    train_dataset = data["train"],
    eval_dataset = data["validation"],
    dataset_text_field = "text",
    max_seq_length = 512,
    dataset_num_proc = 2,
    args = training_arguments
)

print(f"Train samples: {len(trainer.train_dataset)}")
print(f"Eval samples: {len(trainer.eval_dataset)}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/250 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Train samples: 2500
Eval samples: 250


### Model Training

In [ ]:
# !pip uninstall -y wandb
# !pip install wandb

### Train the model

#### free memory space

In [24]:
import torch
import gc

# Clear any cached tensors
torch.cuda.empty_cache()
gc.collect()

# Check available memory
print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print(f"Total VRAM: {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

Free VRAM: 11.11 GB
Total VRAM: 15.64 GB


In [25]:
# Run this BEFORE trainer.train()
print(f"model.max_seq_length:     {model.max_seq_length}")
print(f"max_seq_length variable:  {max_seq_length}")
print(f"Free VRAM:                {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

# Check trainer internals
print(f"trainer max_seq_length:   {trainer.args.max_seq_length if hasattr(trainer.args, 'max_seq_length') else 'not set'}")

model.max_seq_length:     512
max_seq_length variable:  512
Free VRAM:                11.11 GB
trainer max_seq_length:   512


In [26]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,500 | Num Epochs = 2 | Total steps = 626
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Epoch,Training Loss,Validation Loss
1,1.619871,1.746909
2,1.593743,1.748291


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=626, training_loss=1.641202135207935, metrics={'train_runtime': 6621.458, 'train_samples_per_second': 0.755, 'train_steps_per_second': 0.095, 'total_flos': 1.096284196821074e+17, 'train_loss': 1.641202135207935})

### Check memory stats after training

In [28]:
start_gpu_memory = round(torch.cuda.memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(torch.cuda.get_device_properties(0).total_memory / 1024 / 1024 / 1024, 3)
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
# print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
# print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

Peak reserved memory = 11.389 GB.
Peak reserved memory for training = 6.996 GB.
Peak reserved memory % of max memory = 78.205 %.
Peak reserved memory for training % of max memory = 48.04 %.


### Testing the model for a sample conversation

In [38]:
from IPython.display import display, Markdown

FastLanguageModel.for_inference(model)

article = """
Hi ,

I came across your profile and wanted to reach out. I'm Aman — an AI Engineer specializing in LLM fine-tuning, full-stack AI applications, and data analytics. In my current internship at HumanizeIQ.ai I've built Superset dashboards and FastAPI backends, and on the side I've shipped a production AI app (botzcoder.com) using Mistral-7B, Next.js, and PostgreSQL.

I'm actively looking for AI engineer role opportunities and believe my end-to-end ML and analytics experience aligns well with what your team is building.

Happy to share my resume if relevant. Would love to connect!

Best regards,
Aman Kumar Srivastava
Email - aman.apk01@gmail.com
linkedin - https://www.linkedin.com/in/iconic-aman/
""" 

prompt = f"""###Human: Read the following document carefully and provide a concise summary.
Rules:
- Extract only the most important and UNIQUE points
- Each point must be different, no repetition
- Maximum 8 bullet points
- Each point starts with •
- End each point with a period
- include URLs or references if they are mentioned in the document. Don't create any url

Document:
{article}

###Assistant:
- """

inputs = tokenizer(
    [prompt],
    return_tensors = "pt",
    truncation = True,
    max_length = 512,
).to("cuda")

outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 300,
    use_cache = True,
    do_sample = False,
    repetition_penalty = 1.3,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "###Assistant:" in response:
    summary = response.split("###Assistant:")[-1].strip()
else:
    summary = response

display(Markdown(f"### Summary:\n{summary}"))

### Summary:
- 23 year old Indian man seeking job as AI engineer .
- Has worked on projects like botzcoder.com which uses Mistral-7B, Next.js, and PostgreSQL .
- Currently interning at HumanizeIQ.ai where he has used Superset dashboards and FullStackAI apps .
- Looking for similar roles that match his skills .

### Save the model

- local directory

In [30]:
## save locally
import os

model.save_pretrained("outputs/mistral-7b-v1")
tokenizer.save_pretrained("outputs/mistral-7b-v1")
print("Saved locally as v1 ✅")

Saved locally as v1 ✅


- on HuggingFace

In [32]:
repo_id = "aman012/mistral-7b-instruct-v0.3-bnb-4bit-200-v1"
model.push_to_hub(
    repo_id=repo_id,
    commit_message="Initial model push with 2500 data",
    private=False,
)

README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  557kB /  168MB            

Saved model to https://huggingface.co/aman012/mistral-7b-instruct-v0.3-bnb-4bit-200-v1
